# Quantile forest conditional weights

Use native quantile prediction and separately declared conditional-distribution weights; calibration is evaluated on held-out rows.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_pinball_loss
from forestgap import (
    Proximity, conditional_distribution_weights, conditional_quantiles,
)

X, y = make_regression(n_samples=N_SAMPLES, n_features=6, noise=8, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=RANDOM_STATE)
try:
    from quantile_forest import RandomForestQuantileRegressor
except ImportError:
    RandomForestQuantileRegressor = None

if RandomForestQuantileRegressor is None:
    result = {"status": "skipped", "reason": "install forestgap[quantile]"}
else:
    forest = RandomForestQuantileRegressor(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE).fit(X_train, y_train)
    proximity = Proximity(forest, relation="conditional").attach(X_train, y_train)
    quantiles = [0.1, 0.5, 0.9]
    weights = conditional_distribution_weights(proximity, X_test)
    weighted = conditional_quantiles(proximity, y_train, quantiles, X=X_test)
    native = forest.predict(X_test, quantiles=quantiles)
    result = {
        "status": "ok",
        "weight_row_mass": np.asarray(weights.sum(axis=1)).ravel().round(8).tolist()[:5],
        "weighted_80pct_coverage": float(np.mean((y_test >= weighted[:, 0]) & (y_test <= weighted[:, 2]))),
        "native_80pct_coverage": float(np.mean((y_test >= native[:, 0]) & (y_test <= native[:, 2]))),
        "weighted_pinball": float(np.mean([
            mean_pinball_loss(y_test, weighted[:, i], alpha=q) for i, q in enumerate(quantiles)
        ])),
        "native_pinball": float(np.mean([
            mean_pinball_loss(y_test, native[:, i], alpha=q) for i, q in enumerate(quantiles)
        ])),
        "median_mean_absolute_difference": float(np.mean(np.abs(weighted[:, 1] - native[:, 1]))),
    }
result

{'status': 'ok',
 'weight_row_mass': [1.0, 1.0, 1.0, 1.0, 1.0],
 'weighted_80pct_coverage': 0.85,
 'native_80pct_coverage': 0.75,
 'weighted_pinball': 29.379256488381895,
 'native_pinball': 30.85194389519913,
 'median_mean_absolute_difference': 14.649505538689919}